# victoria_geo Dataset — Profiling & Data Quality Assessment

**Project:** FireFusion — Data Engineering Stream
**Dataset:** Victorian Suburbs Latitude and Longitude (source: Wildlife Victoria)

In [1]:
import pandas as pd
import json

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

with open('victoria_suburbs_geo.json', 'r') as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)
print(f"Loaded {len(df)} records")
df.head()


Loaded 3540 records


,latitude,longitude,electoraterating,suburb,postcode,state
0,-37.815207,144.963937,Inner Metropolitan,MELBOURNE,3000,VIC
1,-37.813628,144.963058,,MELBOURNE,3001,VIC
2,-37.816144,144.980459,Inner Metropolitan,EAST MELBOURNE,3002,VIC
3,-37.811450,144.925397,Inner Metropolitan,WEST MELBOURNE,3003,VIC
4,-37.830158,144.980459,Inner Metropolitan,MELBOURNE,3004,VIC


## 1. Dataset Structure & Data Types

In [2]:
print("Shape:", df.shape)
print()
print(df.dtypes)
print()
df.describe(include='all')


Shape: (3540, 6)

latitude            float64
longitude           float64
electoraterating     object
suburb               object
postcode             object
state                object
dtype: object



,latitude,longitude,electoraterating,suburb,postcode,state
count,3540.000000,3540.000000,3540,3540,3540,3540
unique,NaN,NaN,5,3351,749,1
top,NaN,NaN,Rural,GPO MELBOURNE,3352,VIC
freq,NaN,NaN,2610,12,62,3540
mean,-37.199978,144.345319,NaN,NaN,NaN,NaN
std,3.018424,8.711937,NaN,NaN,NaN,NaN
min,-39.031723,-122.235830,NaN,NaN,NaN,NaN
25%,-37.940550,143.574639,NaN,NaN,NaN,NaN
50%,-37.549750,144.836727,NaN,NaN,NaN,NaN
75%,-36.696805,145.651105,NaN,NaN,NaN,NaN


## 2. Missing Values

The source documentation notes that `electoraterating` may contain blanks. Checking all columns for both `NaN`/`None` and empty-string blanks (since blanks in this dataset are stored as `""`, not `null`).

In [3]:
missing_summary = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'empty_string_count': (df.astype(str) == '').sum(),
})
missing_summary['total_missing'] = missing_summary['null_count'] + missing_summary['empty_string_count']
missing_summary['pct_missing'] = (missing_summary['total_missing'] / len(df) * 100).round(2)
missing_summary


,null_count,empty_string_count,total_missing,pct_missing
latitude,0,0,0,0.00
longitude,0,0,0,0.00
electoraterating,0,83,83,2.34
suburb,0,0,0,0.00
postcode,0,0,0,0.00
state,0,0,0,0.00


In [4]:
# electoraterating blanks specifically
blank_electorate = df[df['electoraterating'] == '']
print(f"Blank electoraterating: {len(blank_electorate)} rows ({len(blank_electorate)/len(df)*100:.2f}%)")
blank_electorate.head()


Blank electoraterating: 83 rows (2.34%)


,latitude,longitude,electoraterating,suburb,postcode,state
1,-37.813628,144.963058,,MELBOURNE,3001,VIC
7,-37.822674,144.953839,,WORLD TRADE CENTRE,3005,VIC
11,-37.798346,144.960974,,UNIVERSITY OF MELBOURNE,3010,VIC
100,-37.726300,144.897000,,CROSS KEYS,3041,VIC
125,-37.798915,144.956170,,ROYAL MELBOURNE HOSPITAL,3050,VIC


## 3. Duplicate Records

In [5]:
exact_dupes = df[df.duplicated(keep=False)]
print(f"Exact duplicate rows: {df.duplicated().sum()}")
exact_dupes.sort_values(list(df.columns))


Exact duplicate rows: 1


,latitude,longitude,electoraterating,suburb,postcode,state
2752,-37.6267,145.286706,Rural,SMITHS GULLY,3760,VIC
2753,-37.6267,145.286706,Rural,SMITHS GULLY,3760,VIC


## 4. Postcode & Suburb Consistency

The source doc notes that postcode-to-suburb is not 1:1 (a postcode can map to multiple suburbs, and a suburb can appear under multiple postcode records). Checking this is expected behaviour rather than a data quality issue.

In [6]:
postcode_counts = df['postcode'].value_counts()
multi_suburb_postcodes = postcode_counts[postcode_counts > 1]
print(f"Postcodes linked to multiple suburb records: {len(multi_suburb_postcodes)}")
multi_suburb_postcodes.head(10)


Postcodes linked to multiple suburb records: 523


postcode
3352    62
3401    48
3551    41
3691    37
3875    35
3678    35
3478    34
3825    31
3579    30
3477    30
Name: count, dtype: int64

In [7]:
suburb_counts = df['suburb'].value_counts()
multi_postcode_suburbs = suburb_counts[suburb_counts > 1]
print(f"Suburb names appearing more than once: {len(multi_postcode_suburbs)}")
multi_postcode_suburbs.head(10)


Suburb names appearing more than once: 171


suburb
GPO MELBOURNE    12
MELBOURNE         3
BIG HILL          3
HORSHAM           3
BENALLA           3
HAPPY VALLEY      3
SPRINGFIELD       3
PLUMPTON          3
GOLDEN POINT      3
FLOWERDALE        2
Name: count, dtype: int64

In [8]:
# postcode format check — should always be a 4-digit string
bad_format = df[~df['postcode'].astype(str).str.match(r'^\d{4}$')]
print(f"Postcodes not matching 4-digit format: {len(bad_format)}")
bad_format


Postcodes not matching 4-digit format: 0


,latitude,longitude,electoraterating,suburb,postcode,state


In [9]:
# state column — should be exclusively VIC
print(df['state'].value_counts())


state
VIC    3540
Name: count, dtype: int64


## 5. Geographic Validity Check

Victoria's approximate bounding box: latitude **-39.2 to -33.9**, longitude **140.9 to 150.0**.
Flagging any record whose coordinates fall outside this range as suspicious.

In [10]:
VIC_LAT_MIN, VIC_LAT_MAX = -39.2, -33.9
VIC_LON_MIN, VIC_LON_MAX = 140.9, 150.0

out_of_bounds = df[
    (df['latitude'] < VIC_LAT_MIN) | (df['latitude'] > VIC_LAT_MAX) |
    (df['longitude'] < VIC_LON_MIN) | (df['longitude'] > VIC_LON_MAX)
]
print(f"Records outside expected VIC bounding box: {len(out_of_bounds)}")
out_of_bounds


Records outside expected VIC bounding box: 11


,latitude,longitude,electoraterating,suburb,postcode,state
1445,-34.584100,150.434000,Rural,YARRUNGA,3415,VIC
1699,34.111200,-82.867100,Rural,ELBERTON,3478,VIC
2007,-34.026460,150.850700,Rural,MINTO,3551,VIC
2334,-26.223900,28.300900,Rural,WATTVILLE,3646,VIC
2548,40.178489,-122.235830,Rural,RED BLUFF,3695,VIC
2563,-27.459580,152.663801,Rural,FERNVALE,3701,VIC
3197,-32.550000,151.616667,Rural,MARTINS CREEK,3888,VIC
3334,30.461586,-87.314974,Rural,BELLVIEW,3945,VIC
3404,-35.135000,138.477000,Rural,CHRISTIES,3962,VIC
3489,49.180500,-2.103230,,ST HELIER,3989,VIC


In [11]:
# Of these, which fall clearly outside Australia entirely (different hemisphere / continent)
# vs just outside a strict VIC box while still being within Australia?
clearly_wrong = out_of_bounds[
    (out_of_bounds['latitude'] > 0) |  # positive latitude = Northern Hemisphere, impossible for VIC
    (out_of_bounds['longitude'] < 100)  # far west of Australia entirely
]
print(f"Outside Australia entirely: {len(clearly_wrong)}")
clearly_wrong

Outside Australia entirely: 6


,latitude,longitude,electoraterating,suburb,postcode,state
1699,34.111200,-82.867100,Rural,ELBERTON,3478,VIC
2334,-26.223900,28.300900,Rural,WATTVILLE,3646,VIC
2548,40.178489,-122.235830,Rural,RED BLUFF,3695,VIC
3334,30.461586,-87.314974,Rural,BELLVIEW,3945,VIC
3489,49.180500,-2.103230,,ST HELIER,3989,VIC
3516,40.712743,-74.013379,,GPO WORLD TRADE CENTRE,8005,VIC


In [12]:
# The remainder are still within Australia's bounds, just outside VIC's box —
# these turned out NOT to be legitimate rural VIC records (see root cause analysis below),
# but interstate coordinate collisions instead.
marginal = out_of_bounds.drop(clearly_wrong.index)
print(f"Within Australia but outside VIC bounding box: {len(marginal)}")
marginal

Within Australia but outside VIC bounding box: 5


,latitude,longitude,electoraterating,suburb,postcode,state
1445,-34.58410,150.434000,Rural,YARRUNGA,3415,VIC
2007,-34.02646,150.850700,Rural,MINTO,3551,VIC
2563,-27.45958,152.663801,Rural,FERNVALE,3701,VIC
3197,-32.55000,151.616667,Rural,MARTINS CREEK,3888,VIC
3404,-35.13500,138.477000,Rural,CHRISTIES,3962,VIC


### 5.1 Root Cause Analysis: Suburb Name Collisions

All 11 out-of-bounds records above were cross-checked manually against Australian postcode/locality records. In every case checked, **the suburb name and postcode ARE a legitimate Victorian locality** — the postcode falls within a valid VIC postcode range and corresponds to a real place. The issue is that the recorded coordinate does not point to that Victorian locality at all; it points to a **different, same-named place** — either interstate or overseas.

Verified examples:

| Suburb (VIC postcode) | Recorded coordinates | Actual place at that coordinate |
|---|---|---|
| ELBERTON (3478) | 34.1112, -82.8671 | Elberton, South Carolina, USA |
| WATTVILLE (3646) | -26.2239, 28.3009 | Wattville, South Africa |
| RED BLUFF (3695) | 40.1785, -122.2358 | Red Bluff, California, USA |
| BELLVIEW (3945) | 30.4616, -87.3150 | Bellview, Alabama, USA |
| ST HELIER (3989) | 49.1805, -2.1032 | St Helier, Jersey (Channel Islands) |
| GPO WORLD TRADE CENTRE (8005) | 40.7127, -74.0134 | World Trade Center, New York, USA |
| MINTO (3551) | -34.0265, 150.8507 | Minto, NSW (confirmed — matches the real Minto railway station coordinates almost exactly) |

The remaining four "within Australia" records (YARRUNGA, FERNVALE, MARTINS CREEK, CHRISTIES) follow the same pattern — each has a same-named locality in another state (NSW, QLD, or SA) at approximately the recorded coordinate.

**Conclusion:** this is not random bad data — it looks like the original geocoding process matched suburb names to the wrong real-world place whenever the same name exists elsewhere in Australia or overseas, without checking state/country context.

## 6. Summary of Findings

- **Rows / columns:** 3,540 rows, 6 columns — matches source documentation exactly.
- **Missing values:** No missing `latitude`/`longitude`. 83 rows (2.34%) have a blank `electoraterating` value.
- **Duplicates:** 1 exact duplicate row found.
- **Postcode/suburb consistency:** Multiple suburbs sharing a postcode, and suburbs repeating across postcode records, occurs frequently — this is expected behaviour per the source documentation, not a data quality fault.
- **Postcode format / state column:** All postcodes are valid 4-digit strings; `state` is consistently `VIC` across all records — no issues.
- **Geographic validity:** 11 records have coordinates that don't match their stated VIC suburb/postcode. All 11 are legitimate Victorian localities on paper — the root cause is a **name-collision geocoding error**, where the coordinate assigned was pulled from a same-named place in another state or country instead of the correct Victorian location (verified directly for 7 of the 11 — see Section 5.1). This means the fix is re-geocoding these specific records, not dropping them, since the suburbs themselves are real and needed in the reference table.